# NASA Mission Intelligence: Modular RAG Walkthrough

This notebook explains and verifies the current RAG system without duplicating its implementation.

- The reusable logic lives in the project’s `.py` modules.
- The default path performs cleaning, chunking, database inspection, and result analysis without paid API calls.
- Set `RUN_LIVE_DEMO = True` only when you want one live retrieval-and-generation example.

## 1. Configuration

All runtime choices are defined once. Passing an explicit `.env` path also avoids `python-dotenv` lookup errors in notebooks and standard-input sessions.

In [ ]:
from pathlib import Path
import json
import os

import chromadb
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from batch_evaluation import EVALUATION_METRICS, load_test_questions
from embedding_pipeline import ChromaEmbeddingPipelineTextOnly
from llm_client import DEFAULT_GENERATOR_MODEL, generate_response
from nasa_text_cleaners import build_all_nasa_dataframes
from rag_client import (
    discover_chroma_backends,
    format_context,
    initialize_rag_system,
    retrieve_documents,
)
from ragas_evaluator import DEFAULT_EVALUATOR_MODEL

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data_text"
CHROMA_DIR = PROJECT_DIR / "chroma_db_openai"
COLLECTION_NAME = "nasa_space_missions_text"
RESULTS_PATH = PROJECT_DIR / "evaluation_results_2026-07-28.json"
QUESTIONS_PATH = PROJECT_DIR / "test_questions.json"

EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 5
RUN_LIVE_DEMO = False

load_dotenv(dotenv_path=PROJECT_DIR / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

print(
    {
        "generator_model": DEFAULT_GENERATOR_MODEL,
        "evaluator_model": DEFAULT_EVALUATOR_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "live_demo_enabled": RUN_LIVE_DEMO,
        "api_key_loaded": bool(OPENAI_API_KEY),
    }
)

## 2. Clean the NASA source text

`build_all_nasa_dataframes` applies the current OCR/ASR cleaners and preserves mission, source-file, and provenance metadata. This step is local and makes no API calls.

In [ ]:
reports, transcripts, all_data = build_all_nasa_dataframes(DATA_DIR)

def count_source_files(frame):
    return frame["metadata"].map(
        lambda metadata: metadata["source_path"]
    ).nunique()

cleaning_summary = pd.DataFrame(
    [
        {
            "dataset": "reports",
            "records": len(reports),
            "source_files": count_source_files(reports),
        },
        {
            "dataset": "transcripts",
            "records": len(transcripts),
            "source_files": count_source_files(transcripts),
        },
        {
            "dataset": "combined",
            "records": len(all_data),
            "source_files": count_source_files(all_data),
        },
    ]
)
display(cleaning_summary)

## 3. Audit chunking offline

The pipeline uses a uniform maximum of 500 tokens with a target overlap of 100 tokens. For this offline audit, `object.__new__` bypasses the class constructor because the constructor opens OpenAI and Chroma clients; only the pipeline’s pure chunking methods are used here.

In [ ]:
offline_chunker = object.__new__(ChromaEmbeddingPipelineTextOnly)
offline_chunker.chunk_size = CHUNK_SIZE
offline_chunker.chunk_overlap = CHUNK_OVERLAP

chunks_by_file = offline_chunker.chunk_cleaned_records_by_file(all_data)
chunk_rows = []
for file_path, chunks in chunks_by_file.items():
    for document, metadata in chunks:
        chunk_rows.append(
            {
                "id": offline_chunker.generate_document_id(
                    file_path, metadata
                ),
                "document": document,
                "file_path": str(file_path),
                "mission": metadata["mission"],
                "source_type": metadata["source_type"],
                "chunk_index": metadata["chunk_index"],
                "token_count": metadata["token_count"],
                "metadata": metadata,
            }
        )

corpus_df = pd.DataFrame(chunk_rows)

assert not corpus_df.empty
assert corpus_df["id"].is_unique
assert corpus_df["token_count"].max() <= CHUNK_SIZE
assert corpus_df["mission"].notna().all()
assert corpus_df["file_path"].notna().all()

chunk_summary = (
    corpus_df.groupby(["mission", "source_type"])
    .agg(chunks=("id", "count"), source_files=("file_path", "nunique"))
    .reset_index()
)
display(chunk_summary)
print(
    f"Prepared {len(corpus_df):,} chunks from "
    f"{corpus_df['file_path'].nunique()} source files; "
    f"maximum chunk length = {corpus_df['token_count'].max()} tokens."
)

## 4. Inspect the persisted Chroma collection

This cell discovers an existing collection and reads its count. It does not create a database, generate embeddings, or modify stored records.

In [ ]:
backends = discover_chroma_backends()
backend_key = f"{CHROMA_DIR.name}:{COLLECTION_NAME}"
backend = backends.get(backend_key)
stored_collection = None
stored_chunk_count = None

if backend is None:
    print(
        "Current Chroma collection is not present locally. "
        "The notebook will continue in offline report mode."
    )
else:
    chroma_client = chromadb.PersistentClient(
        path=backend["directory"]
    )
    stored_collection = chroma_client.get_collection(
        name=backend["collection_name"]
    )
    stored_chunk_count = stored_collection.count()
    print(f"Backend: {backend['display_name']}")

display(
    pd.DataFrame(
        [
            {
                "prepared_chunks": len(corpus_df),
                "stored_chunks": stored_chunk_count,
                "counts_match": (
                    stored_chunk_count == len(corpus_df)
                    if stored_chunk_count is not None
                    else None
                ),
            }
        ]
    )
)

## 5. Optional live retrieval and grounded generation

Retrieval embeds the question and generation calls the chat model, so both can incur API usage. The function delegates to `rag_client.py` and `llm_client.py`; the default flag keeps it uncalled.

In [ ]:
test_questions = load_test_questions(QUESTIONS_PATH)
demo_question = test_questions[0]

def run_live_demo(question_record):
    if not OPENAI_API_KEY:
        raise RuntimeError(
            "OPENAI_API_KEY is required for the live demo."
        )
    if backend is None:
        raise RuntimeError(
            "Build the current Chroma collection before the live demo."
        )

    collection, initialized, error = initialize_rag_system(
        chroma_dir=str(CHROMA_DIR),
        collection_name=COLLECTION_NAME,
        openai_api_key=OPENAI_API_KEY,
        openai_base_url=OPENAI_BASE_URL,
        embedding_model=EMBEDDING_MODEL,
    )
    if not initialized:
        raise RuntimeError(error or "RAG initialization failed.")

    retrieval = retrieve_documents(
        collection=collection,
        query=question_record["user_input"],
        n_results=TOP_K,
        mission_filter=question_record["mission"],
    )
    documents = retrieval["documents"][0]
    metadatas = retrieval["metadatas"][0]
    context = format_context(documents, metadatas)
    response = generate_response(
        openai_key=OPENAI_API_KEY,
        user_message=question_record["user_input"],
        context=context,
        conversation_history=[],
        model=DEFAULT_GENERATOR_MODEL,
        openai_base_url=OPENAI_BASE_URL,
    )
    return {
        "question": question_record["user_input"],
        "mission": question_record["mission"],
        "documents": documents,
        "metadatas": metadatas,
        "context": context,
        "response": response,
    }

live_demo = (
    run_live_demo(demo_question)
    if RUN_LIVE_DEMO
    else None
)

if live_demo is None:
    print(
        "Live demo skipped. Set RUN_LIVE_DEMO = True "
        "to make paid retrieval and generation calls."
    )
else:
    print(live_demo["question"])
    print(live_demo["response"])

## 6. Load the reproducible batch evaluation

The tracked JSON contains the completed 17-question run. Loading it is deterministic and free; a new paid run should use `batch_evaluation.py`, which owns the current RAGAS configuration and async client lifecycle.

The six metrics answer different questions:

| Metric | What it checks |
|---|---|
| Response relevancy | Does the answer address the question? |
| Faithfulness | Are answer claims supported by retrieved text? |
| Context precision | How much retrieved text was useful? |
| Context recall | How much reference evidence was retrieved? |
| Factual correctness | Does the answer agree with the reference? |
| Retrieval F1 | Harmonic balance of context precision and recall |

Example: if five chunks are retrieved but only two contain Apollo 13 evidence, context precision can be low even when the final cited answer is faithful.

In [ ]:
evaluation_report = json.loads(
    RESULTS_PATH.read_text(encoding="utf-8")
)
results_df = pd.DataFrame(evaluation_report["results"])

question_ids = {question["id"] for question in test_questions}
result_ids = set(results_df["id"])
assert question_ids == result_ids
assert evaluation_report["summary"]["failed_question_count"] == 0
assert set(EVALUATION_METRICS).issubset(results_df.columns)

metric_summary = (
    pd.DataFrame(evaluation_report["summary"]["metrics"])
    .T.rename(columns={"minimum": "min", "maximum": "max"})
    [["count", "mean", "min", "max"]]
)

print(evaluation_report["configuration"])
display(metric_summary.round(3))

## 7. Inspect question-level strengths and weaknesses

Aggregate means can hide failures on individual questions. The first table shows every score; the second highlights the lowest retrieval-F1 cases for corpus or retrieval improvement.

In [ ]:
result_columns = [
    "id",
    "mission",
    "category",
    *EVALUATION_METRICS,
]
display(results_df[result_columns].round(3))

weakest_retrieval = (
    results_df.nsmallest(5, "retrieval_f1")[
        [
            "id",
            "mission",
            "category",
            "context_precision",
            "context_recall",
            "retrieval_f1",
        ]
    ]
)
display(weakest_retrieval.round(3))

## Conclusion

The current pipeline is operational across all 17 evaluation questions. Response relevancy and faithfulness are stronger than context recall and factual correctness, so the next quality work should focus on retrieval coverage and corpus evidence—not on adding more notebook code.

The notebook is now a thin teaching and verification layer: source cleaning, chunking, retrieval, generation, and evaluation remain maintained in their dedicated Python modules.